# Práctica: Compromiso Sesgo–Varianza con Regresión Polinómica - OLS + 10-Fold CV

**Objetivo:** Para varios datasets, ajustar regresiones polinómicas de grado 1 a 15 y estudiar el compromiso sesgo/varianza comparando el error de entrenamiento y el error de validación (estimado con 10-Fold Cross-Validation).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


## Funciones auxiliares

- Construcción de la matriz de diseño polinómica
- K-Fold manual
- MSE
- 10-Fold CV para barrer grados 1..15
- Ajuste final + p-values

In [ ]:
def make_poly_design(x: np.ndarray, degree: int, add_intercept: bool = True) -> np.ndarray:
    x = np.asarray(x).reshape(-1)
    cols = [x**k for k in range(1, degree + 1)]
    X = np.column_stack(cols) if cols else np.empty((len(x), 0))
    if add_intercept:
        X = sm.add_constant(X, has_constant="add")
    return X

def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return float(np.mean((y_true - y_pred) ** 2))

def kfold_indices(n: int, k: int = 10, seed: int = 123):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        yield train_idx, val_idx

def cv_poly_ols(x, y, degrees=range(1, 16), k=10, seed=123):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)
    n = len(y)

    rows = []
    for deg in degrees:
        train_mses, val_mses = [], []
        for tr, va in kfold_indices(n, k=k, seed=seed):
            Xtr = make_poly_design(x[tr], deg, add_intercept=True)
            Xva = make_poly_design(x[va], deg, add_intercept=True)

            model = sm.OLS(y[tr], Xtr).fit()

            yhat_tr = model.predict(Xtr)
            yhat_va = model.predict(Xva)

            train_mses.append(mse(y[tr], yhat_tr))
            val_mses.append(mse(y[va], yhat_va))

        rows.append({
            "degree": int(deg),
            "mse_train_mean": float(np.mean(train_mses)),
            "mse_train_std": float(np.std(train_mses, ddof=1)),
            "mse_val_mean": float(np.mean(val_mses)),
            "mse_val_std": float(np.std(val_mses, ddof=1)),
        })

    return pd.DataFrame(rows)

def choose_degree(df_cv: pd.DataFrame) -> int:
    return int(df_cv.loc[df_cv["mse_val_mean"].idxmin(), "degree"])

def fit_full_and_pvalues(x, y, degrees=range(1, 16)):
    x = np.asarray(x).reshape(-1)
    y = np.asarray(y).reshape(-1)

    rows = []
    for deg in degrees:
        X = make_poly_design(x, deg, add_intercept=True)
        model = sm.OLS(y, X).fit()

        coefs = model.params
        pvals = model.pvalues

        coef_str = ",".join([f"{c:.6g}" for c in coefs])
        pval_str = ",".join([f"{p:.6g}" for p in pvals])

        rows.append({
            "degree": int(deg),
            "coefficients": coef_str,
            "pvalues": pval_str,
            "pvalue_mean": float(np.mean(pvals)),
        })
    return pd.DataFrame(rows)

def infer_xy_columns(df: pd.DataFrame):
    cols = list(df.columns)

    # Caso típico
    if 'x' in cols and 'y' in cols:
        return 'x', 'y'
    if 'X' in cols and 'Y' in cols:
        return 'X', 'Y'

    if df.shape[1] == 2:
        return cols[0], cols[1]

    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    if len(num_cols) >= 2:
        return num_cols[0], num_cols[-1]

    return cols[0], cols[-1]

In [ ]:
dataset_files = [
    "dataset_1.csv",
    "dataset_2.csv",
    "dataset_3.csv",
    "dataset_4.csv",
    "dataset_5.csv",
    "dataset_6.csv",
]

DEGREES = range(1, 16)
K_FOLDS = 10
SEED = 11


## Análisis por dataset

Para dataset:
1. Exploratorio básico
2. 10-fold CV para grados 1..15
3. Curvas MSE train/val
4. Selección del grado óptimo
5. Ajuste final y gráfica
6. Extra: tabla coef/p-values y gráfica p-value medio


In [ ]:
all_summaries = []

for fname in dataset_files:
    print("========================")
    print(f"DATASET: {fname}")
    print("========================")

    df = pd.read_csv("../data/" + fname)
    print("\nPrimeras filas:")
    display(df.head())

    print("\nDescripción:")
    display(df.describe(include='all'))

    print("\nValores perdidos por columna:")
    display(df.isna().sum())

    df = df.dropna().copy()

    x_col, y_col = infer_xy_columns(df)
    print(f"\nColumnas elegidas: x = '{x_col}', y = '{y_col}'")

    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()

    plt.figure(figsize=(6,4))
    plt.scatter(x, y, s=18, alpha=0.8)
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"{fname}: scatter")
    plt.grid(True, alpha=0.2)
    plt.show()

    df_cv = cv_poly_ols(x, y, degrees=DEGREES, k=K_FOLDS, seed=SEED)
    best_deg = choose_degree(df_cv)

    print("\nResultados CV (MSE):")
    display(df_cv)
    print(f"\n=> Grado óptimo - mínimo MSE validación: {best_deg}")

    plt.figure(figsize=(8,4.5))
    plt.plot(df_cv['degree'], df_cv['mse_train_mean'], marker='o', label='Train MSE (mean)')
    plt.plot(df_cv['degree'], df_cv['mse_val_mean'], marker='o', label='Val MSE (mean)')
    plt.fill_between(df_cv['degree'],
                     df_cv['mse_val_mean']-df_cv['mse_val_std'],
                     df_cv['mse_val_mean']+df_cv['mse_val_std'],
                     alpha=0.15, label='Val ±1 std (folds)')
    plt.axvline(best_deg, color='k', linestyle='--', alpha=0.7, label=f'Best deg={best_deg}')
    plt.xlabel('Polynomial degree')
    plt.ylabel('MSE')
    plt.title(f"{fname}: Bias/Variance via 10-Fold CV")
    plt.legend()
    plt.show()

    X_full = make_poly_design(x, best_deg, add_intercept=True)
    model_full = sm.OLS(y, X_full).fit()
    print("\nResumen OLS del ajuste final:")
    print(model_full.summary())

    order = np.argsort(x)
    x_sorted = x[order]
    X_sorted = make_poly_design(x_sorted, best_deg, add_intercept=True)
    yhat_sorted = model_full.predict(X_sorted)

    plt.figure(figsize=(8,4.5))
    plt.scatter(x, y, s=18, alpha=0.7, label='data')
    plt.plot(x_sorted, yhat_sorted, color='crimson', lw=2.2, label=f'poly degree {best_deg}')
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"{fname}: Final model (degree {best_deg})")
    plt.legend()
    plt.show()

    best_row = df_cv[df_cv['degree'] == best_deg].iloc[0]
    msg = []
    msg.append(f"Para {fname}, el grado óptimo por 10-fold CV es {best_deg}.")
    msg.append(f"MSE(train)≈{best_row['mse_train_mean']:.4g} y MSE(val)≈{best_row['mse_val_mean']:.4g}.")

    print("\nConclusión sesgo/varianza:")
    print("- " + "\n- ".join(msg))

    df_p = fit_full_and_pvalues(x, y, degrees=DEGREES)
    print("\nTabla coeficientes y p-values:")
    display(df_p)

    plt.figure(figsize=(8,4.5))
    plt.plot(df_p['degree'], df_p['pvalue_mean'], marker='o')
    plt.axvline(best_deg, color='k', linestyle='--', alpha=0.7)
    plt.xlabel('Polynomial degree')
    plt.ylabel('Mean p-value (coefficients)')
    plt.title(f"{fname}: Mean p-value vs Degree")
    plt.show()

    all_summaries.append({
        "dataset": fname,
        "x_col": x_col,
        "y_col": y_col,
        "best_degree": best_deg,
        "mse_train_at_best": float(best_row['mse_train_mean']),
        "mse_val_at_best": float(best_row['mse_val_mean']),
    })

## Resumen agrupando datasets

In [ ]:
df_summary = pd.DataFrame(all_summaries)
display(df_summary)

plt.figure(figsize=(8,4.5))
plt.bar(df_summary['dataset'], df_summary['best_degree'])
plt.ylabel('Best polynomial degree')
plt.title('Best degree by dataset (10-Fold CV)')
plt.xticks(rotation=0)
plt.show()

plt.figure(figsize=(8,4.5))
plt.plot(df_summary['dataset'], df_summary['mse_train_at_best'], marker='o', label='Train MSE at best deg')
plt.plot(df_summary['dataset'], df_summary['mse_val_at_best'], marker='o', label='Val MSE at best deg')
plt.ylabel('MSE')
plt.title('Train vs Val MSE at chosen degree')
plt.legend()
plt.show()


## Resultados

Tras aplicar las condiciones del enunciado (regresión polinómica OLS con validación cruzada 10-Fold y evaluación mediante MSE), se determinaron los grados polinómicos óptimos para cada conjunto de datos seleccionando el grado que minimiza el MSE de validación, por ser la mejor estimación del error de generalización.

Grado óptimo por dataset

Dataset 1:
El grado óptimo es 5 (MSE_train ≈ 23.71, MSE_val ≈ 24.00). En grados inferiores se aprecia sesgo (modelo demasiado simple). A partir del grado 5, el error de validación entra en una meseta; aumentar la complejidad no aporta mejoras relevantes y solo incrementa el coste.

Dataset 2:
El grado óptimo es 5 (MSE_train ≈ 23.71, MSE_val ≈ 24.00). El comportamiento es similar al del dataset 1: los grados bajos muestran subajuste y, a partir del grado 5, la validación no mejora de manera sostenida.

Dataset 3:
El grado óptimo es 5 (MSE_train ≈ 23.71, MSE_val ≈ 24.00). La cercanía entre el error de entrenamiento y validación en el grado elegido sugiere un ajuste razonable y ausencia de sobreajuste marcado.

Dataset 4:
El grado óptimo es 9 (MSE_train ≈ 0.243, MSE_val ≈ 0.248). Este conjunto requiere mayor flexibilidad por su relación no lineal. El error de validación disminuye de forma consistente hasta el grado 9; a partir de ahí, la mejora posterior es marginal.

Dataset 5:
El grado óptimo es 2 (MSE_train ≈ 20076, MSE_val ≈ 20159). Los errores son muy elevados y la dispersión es grande, lo que sugiere una fuerte presencia de ruido y/o outliers. Incrementar la complejidad no mejora la generalización, por lo que se elige un modelo simple y estable.

Dataset 6:
El grado óptimo es 7 (MSE_train ≈ 3.488, MSE_val ≈ 3.557). Se observa mejora hasta grados intermedios. A partir del grado 7, el error de entrenamiento sigue bajando pero el de validación se estanca, señalando el inicio de una zona dominada por varianza.

Fundamentos teóricos: sesgo y varianza

El comportamiento observado en las curvas de MSE frente al grado refleja el compromiso sesgo–varianza. El sesgo predomina en modelos demasiado simples y se identifica por errores de entrenamiento y validación simultáneamente altos. La varianza aparece en modelos demasiado complejos: el error de entrenamiento baja mucho, pero el de validación se estanca o empeora. El grado seleccionado en cada dataset corresponde al punto donde se minimiza el MSE de validación, logrando el mejor equilibrio entre capacidad del modelo y generalización.

Análisis de significancia (p-values) y comparación con sesgo–varianza

Se analizó la utilidad de los coeficientes mediante tablas de p-values y la gráfica de p-value medio frente al grado. En general, al aumentar el grado crece el número de términos con p-values altos, lo que indica menor evidencia estadística de aportación real de muchos coeficientes. Esto suele coincidir con regiones en las que el MSE de validación deja de mejorar. Aunque el criterio principal de selección ha sido el MSE por validación cruzada, los p-values apoyan la idea de que añadir complejidad innecesaria introduce inestabilidad sin mejorar la predicción.

Conclusión global

En los datasets 1, 2 y 3 se observa un patrón muy similar en las curvas de MSE y por eso el grado óptimo coincide. Esto sugiere que estos conjuntos podrían compartir una estructura subyacente comparable (relación x–y y nivel de ruido similares), haciendo que la complejidad óptima requerida sea prácticamente la misma. Sin embargo, no existe un grado polinómico universal; el grado óptimo depende de la forma de la relación x–y y del nivel de ruido de cada dataset. En esta práctica, la secuencia óptima ha sido (5, 5, 5, 9, 2, 7). La validación cruzada 10-Fold ha permitido identificar el punto en el que aumentar la complejidad deja de capturar patrón útil y empieza a ajustar ruido, caracterizando el equilibrio entre sesgo y varianza.